# GLASS-JWST NIRSpec Spectral Viewer — Abell 2744

Two viewing modes controlled by `PLOT_MODE` in the Parameters cell:

| Mode | Description |
|---|---|
| `"range"` | Plot a range of spectra (IDX_START–IDX_END), one panel per target, single redshift |
| `"z_quad"` | Plot one **2×2 figure per target** showing the same spectrum at four z-windows simultaneously — useful for visual redshift identification |

**Each `_spec.fits` (produced by msaexp) contains:**
- EXT 0 — Primary HDU (metadata only)
- EXT 1 — 2-D spectral trace image
- EXT 2+ — 1-D extracted spectrum (BINTABLE with WAVE / FLUX / ERR columns)

---


In [ ]:
# ============================================================
#  PARAMETERS  — edit these, then Kernel → Restart & Run All
# ============================================================

# Path to the root of your mastDownload tree
MAST_ROOT = "./mastDownload/HLSP"

# ── Plot mode ────────────────────────────────────────────────
# "range"  → one panel per spectrum, single redshift (original behaviour)
# "z_quad" → one 2×2 figure per spectrum, four z-windows per target
PLOT_MODE = "auto"

# Index range of spectra to plot (inclusive, zero-based)
IDX_START = 0
IDX_END   = 9

# ── z_quad mode: four redshift windows ───────────────────────
# Each entry is (label, z_value).  Edit z values to taste.
Z_WINDOWS = [
    ("Cluster  z ~ 0.31",  0.308),   # Abell 2744 cluster redshift
    ("Low-z   z ~ 1.5",   1.5),      # MgII 2798, [OII] 3727 in PRISM window
    ("Mid-z   z ~ 4.0",   4.0),      # Lyα, CIII], CIV enter NIR grating range
    ("High-z  z ~ 7.0",   7.0),      # High-z GLASS-JWST science targets
]

# ── range mode: single redshift override ─────────────────────
# Set to None to use per-file header value instead.
Z_OVERRIDE = None

# Set True to show the 2-D spectral trace above each 1-D panel (range mode only)
SHOW_2D = False

# ── Emission line annotation ─────────────────────────────────
SHOW_LINES = True

ANNOTATE_GROUPS = {
    "hydrogen",
    "forbidden",
    "agn_uv",
    "agn_coronal",
    "sfr",
}

# ── Mascia+2024 catalog cross-match ──────────────────────────
# Local cache of the GLASS-JWST spectroscopic catalog.
# Cell 9 downloads it from MAST on first run and saves it here.
# Kept in the notebook directory so it survives mastDownload cleanups.
# Set to None to disable catalog lookup entirely.
CATALOG_PATH = "./glass_jwst_cat.fits"

# Set True to save each figure as a PNG
SAVE_PNG = True

# Output directory for saved PNGs
OUTPUT_DIR = "."


## 1 · Imports & display setup

In [ ]:
import glob
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import AsinhNorm
from astropy.io import fits
from astropy.wcs import FITSFixedWarning
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FITSFixedWarning)
warnings.filterwarnings("ignore", category=fits.verify.VerifyWarning)

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

_MAST_ROOT  = Path(MAST_ROOT)
_OUTPUT_DIR = Path(OUTPUT_DIR)

_WAVE_NAMES = ["WAVE", "WAVELENGTH", "LAMBDA"]
_FLUX_NAMES = ["FLUX", "FLUX_CORR", "FNU"]
_ERR_NAMES  = ["ERR",  "UNCERTAINTY", "FLUX_ERR", "ERROR"]

COLS_PER_ROW = 1
FIG_WIDTH    = 16
PANEL_HEIGHT = 3.8

print("Imports OK ✓")


## 2 · Emission line catalogue

Rest-frame wavelengths (Å) for lines routinely targeted by JWST NIRSpec
galaxy and AGN programmes. Lines are grouped into five physical categories
and colour-coded in the plots.

| Group | Colour | Lines |
|---|---|---|
| `hydrogen` | tomato | Lyα, Hα, Hβ, Hγ, Hδ, Paα, Paβ, Paγ, Paδ, Brα, Brβ, Brγ |
| `forbidden` | gold | [OII], [OIII], [NII], [SII], [SIII], [NeIII], [ArIII] |
| `agn_uv` | orchid | NV, CIV, HeII, CIII], MgII, [NeIV], [NeV] |
| `agn_coronal` | deepskyblue | [FeVII], [FeX], [FeXI], [FeXIV] |
| `sfr` | limegreen | classic SFR / BPT diagnostics (Hα, Hβ, [OII]3727, [OIII]5007) |

> Lines are shifted to the **observed** frame using the redshift extracted from
> the FITS header (`REDSHIFT`, `Z_SPEC`, or `Z_PHOT` in that priority order).
> If no redshift is available the annotation is skipped for that spectrum.


In [ ]:
# ---------------------------------------------------------------------------
#  Emission line catalogue
#  (rest-frame wavelengths in Ångströms)
# ---------------------------------------------------------------------------
#
#  References used to compile this list:
#   • Mascia et al. 2024 (GLASS-JWST spectroscopic release)
#   • Tang et al. 2025 (JWST NIRSpec high-ionisation lines)
#   • Osterbrock & Ferland 2006 (Astrophysics of Gaseous Nebulae)
#   • Véron-Cetty & Véron 2006 AGN line atlas
#   • NIST Atomic Spectra Database
# ---------------------------------------------------------------------------

EMISSION_LINES = {

    # ── Hydrogen recombination series ──────────────────────────────────────
    "hydrogen": {
        "color"  : "tomato",
        "lw"     : 0.9,
        "alpha"  : 0.85,
        "lines"  : {
            # Lyman series (UV)
            "Lyα"   : 1215.67,
            "Lyβ"   : 1025.72,
            "Lyγ"   : 972.54,
            # Balmer series (optical)
            "Hα"    : 6562.80,
            "Hβ"    : 4861.33,
            "Hγ"    : 4340.47,
            "Hδ"    : 4101.74,
            "Hε"    : 3970.07,
            "H8"    : 3889.05,
            "H9"    : 3835.39,
            # Paschen series (NIR)
            "Paα"   : 18751.0,
            "Paβ"   : 12818.1,
            "Paγ"   : 10938.1,
            "Paδ"   : 10049.4,
            "Paε"   : 9546.0,
            # Brackett series (NIR)
            "Brα"   : 40522.0,
            "Brβ"   : 26258.0,
            "Brγ"   : 21661.0,
            "Brδ"   : 19446.0,
        },
    },

    # ── Forbidden / collisionally excited lines ────────────────────────────
    "forbidden": {
        "color"  : "gold",
        "lw"     : 0.8,
        "alpha"  : 0.80,
        "lines"  : {
            # Oxygen
            "[OII]3726"  :  3726.03,
            "[OII]3729"  :  3728.82,
            "[OIII]4363" :  4363.21,
            "[OIII]4959" :  4958.92,
            "[OIII]5007" :  5006.84,
            "[OI]6300"   :  6300.30,
            "[OI]6364"   :  6363.78,
            # Nitrogen
            "[NII]5755"  :  5754.64,
            "[NII]6548"  :  6548.05,
            "[NII]6583"  :  6583.45,
            # Sulphur
            "[SII]6716"  :  6716.44,
            "[SII]6731"  :  6730.82,
            "[SIII]9069" :  9068.60,
            "[SIII]9532" :  9531.10,
            # Neon
            "[NeIII]3869":  3868.76,
            "[NeIII]3967":  3967.47,
            "[NeV]3346"  :  3345.83,
            "[NeV]3426"  :  3425.88,
            # Argon
            "[ArIII]7135":  7135.78,
            "[ArIII]7751":  7751.10,
            "[ArIV]4711" :  4711.37,
            "[ArIV]4740" :  4740.17,
            # Helium
            "HeI 5876"   :  5875.67,
            "HeI 6678"   :  6678.15,
            "HeI 7065"   :  7065.22,
            "HeI 10830"  : 10830.34,
        },
    },

    # ── AGN / high-ionisation UV lines ─────────────────────────────────────
    # Seen in Type I/II AGN and "Little Red Dots" with JWST NIRSpec.
    # Key refs: Tang+2025, Scholtz+2023, Mazzolari+2024
    "agn_uv": {
        "color"  : "orchid",
        "lw"     : 1.0,
        "alpha"  : 0.90,
        "lines"  : {
            # UV metal lines
            "NV 1238"    :  1238.82,   # doublet; AGN ionisation tracer
            "NV 1242"    :  1242.80,
            "CIV 1548"   :  1548.20,   # doublet; broad in Type I AGN
            "CIV 1551"   :  1550.78,
            "HeII 1640"  :  1640.42,   # high-ionisation; AGN / Pop III
            "OIII] 1661" :  1660.81,   # semi-forbidden doublet
            "OIII] 1666" :  1666.15,
            "CIII] 1907" :  1906.68,   # semi-forbidden doublet; UV SFR/AGN
            "CIII] 1909" :  1908.73,
            "CII] 2326"  :  2326.11,
            "MgII 2796"  :  2795.53,   # doublet; broad in Type I AGN
            "MgII 2803"  :  2802.71,
            "[NeIV] 2423":  2422.56,   # doublet; AGN narrow line region
            "[NeIV] 2426":  2425.14,
            "[NeV] 3346" :  3345.83,   # high-ionisation; AGN
            "[NeV] 3426" :  3425.88,
            "HeII 4686"  :  4685.68,   # optical HeII; AGN / WR stars
            "[FeII] 1.26μ": 12570.0,   # NIR FeII; AGN / shock tracer
            "[FeII] 1.64μ": 16440.0,
        },
    },

    # ── AGN coronal lines ──────────────────────────────────────────────────
    # Arise in the highly ionised coronal line region close to the AGN.
    # Detectable with JWST NIRSpec in the rest-frame optical/NIR.
    # Refs: Gilli+2010, Murayama+1998, Riffel+2021
    "agn_coronal": {
        "color"  : "deepskyblue",
        "lw"     : 0.8,
        "alpha"  : 0.80,
        "lines"  : {
            "[FeVII] 3586" :  3586.32,
            "[FeVII] 5159" :  5158.89,
            "[FeVII] 5721" :  5720.70,
            "[FeVII] 6087" :  6087.00,
            "[FeX]  6374"  :  6374.51,
            "[FeXI] 7892"  :  7891.80,
            "[FeXIV] 5303" :  5302.86,
            "[SiVI] 1.963μ": 19630.0,
            "[CaVIII] 2.32μ": 23210.0,
            "[SiVII] 2.48μ": 24820.0,
            "[MgVIII] 3.03μ": 30280.0,
            "[MgVII] 5.50μ": 55030.0,
        },
    },

    # ── Classic SFR / BPT diagnostic lines ────────────────────────────────
    # This group is a curated subset of the above for users who only want
    # the handful of lines used in BPT diagrams and SFR calibrations.
    # Displayed in a distinct bright green to stand out.
    "sfr": {
        "color"  : "limegreen",
        "lw"     : 1.2,
        "alpha"  : 0.95,
        "lines"  : {
            "[OII] 3727"  :  3727.43,  # blended doublet centroid
            "Hβ"          :  4861.33,
            "[OIII] 4959" :  4958.92,
            "[OIII] 5007" :  5006.84,
            "Hα"          :  6562.80,
            "[NII] 6583"  :  6583.45,
            "[SII] 6716"  :  6716.44,
            "[SII] 6731"  :  6730.82,
        },
    },
}


# ---------------------------------------------------------------------------
#  Annotation helper
# ---------------------------------------------------------------------------

def annotate_emission_lines(
    ax,
    wave_obs: np.ndarray,
    ylo: float,
    yhi: float,
    redshift: float | None,
    groups: set[str],
) -> None:
    """
    Overplot vertical lines + labels for emission lines that fall within
    the observed wavelength window of this spectrum.

    Parameters
    ----------
    ax        : matplotlib Axes (the 1-D spectrum panel)
    wave_obs  : observed wavelength array in Å
    ylo, yhi  : current y-axis limits (used to position labels)
    redshift  : source redshift; if None the annotation is skipped
    groups    : set of group keys from EMISSION_LINES to draw
    """
    if redshift is None:
        return

    z    = float(redshift)
    wmin = wave_obs.min()
    wmax = wave_obs.max()
    span = yhi - ylo

    # Track x-positions already labelled to stagger overlapping labels
    labelled_x: list[float] = []

    for grp_key in groups:
        if grp_key not in EMISSION_LINES:
            continue
        grp     = EMISSION_LINES[grp_key]
        color   = grp["color"]
        lw      = grp["lw"]
        alpha   = grp["alpha"]

        for name, lam_rest in grp["lines"].items():
            lam_obs = lam_rest * (1.0 + z)
            if not (wmin <= lam_obs <= wmax):
                continue

            # Vertical line
            ax.axvline(lam_obs, color=color, linewidth=lw,
                       alpha=alpha, linestyle="--", zorder=4)

            # Stagger label height to reduce collision
            n_nearby = sum(1 for x in labelled_x if abs(x - lam_obs) < 180)
            y_frac   = 0.97 - 0.12 * (n_nearby % 5)
            y_pos    = ylo + y_frac * span

            ax.text(
                lam_obs, y_pos,
                name,
                color=color,
                fontsize=8,
                rotation=90,
                va="top",
                ha="center",
                alpha=min(alpha + 0.1, 1.0),
                zorder=5,
                clip_on=True,
            )
            labelled_x.append(lam_obs)


def extract_redshift(hdr: fits.Header, filepath=None) -> float | None:
    """
    Return the redshift to use for emission-line annotation.

    Priority order:
      1. Z_OVERRIDE  — if set in parameters cell, always wins
      2. SOURCE_CATALOG (Mascia+2024) — keyed on MSA ID from filename
      3. FITS header keywords: REDSHIFT → Z_SPEC → Z_PHOT → ZNEW → Z
      4. None — annotation skipped for this spectrum
    """
    # 1. Hard-coded override
    try:
        if Z_OVERRIDE is not None:
            z = float(Z_OVERRIDE)
            if 0.0 <= z < 25.0:
                return z
    except (NameError, TypeError, ValueError):
        pass

    # 2. Mascia+2024 catalog
    try:
        if filepath is not None and SOURCE_CATALOG:
            msa_id = extract_msa_id(filepath)
            if msa_id is not None:
                entry = SOURCE_CATALOG.get(msa_id)
                if entry is not None and entry["z"] is not None:
                    return entry["z"]
    except NameError:
        pass  # SOURCE_CATALOG not yet defined

    # 3. FITS header
    for kw in ("REDSHIFT", "Z_SPEC", "Z_PHOT", "ZNEW", "Z"):
        val = hdr.get(kw, None)
        if val is not None:
            try:
                z = float(val)
                if 0.0 <= z < 25.0:
                    return z
            except (TypeError, ValueError):
                pass
    return None


print(f"Emission line catalogue loaded ✓")
print(f"  Groups : {', '.join(EMISSION_LINES.keys())}")
total = sum(len(g['lines']) for g in EMISSION_LINES.values())
print(f"  Total lines defined : {total}")


## 3 · File discovery & inventory

In [ ]:
def find_spec_files(root: Path) -> list[Path]:
    """Recursively find all per-source NIRSpec _spec.fits files."""
    pattern   = str(root / "**" / "*nirspec*spec.fits")
    all_files = sorted(glob.glob(pattern, recursive=True))
    return [Path(f) for f in all_files if "spec-template" not in f]


def classify_files(files: list[Path]) -> dict[str, list[Path]]:
    """Group files by grating/filter optical element token in the filename."""
    groups: dict[str, list[Path]] = {}
    for f in files:
        parts   = f.stem.replace("-", "_").split("_")
        optelem = next(
            (p for p in parts
             if any(g in p for g in ["g140h", "g235h", "g395h", "prism"])),
            "unknown",
        )
        groups.setdefault(optelem, []).append(f)
    return groups


print(f"Searching under: {_MAST_ROOT.resolve()}")
spec_files = find_spec_files(_MAST_ROOT)
groups     = classify_files(spec_files)

if not spec_files:
    display(Markdown(
        "⚠️ **No spectral FITS files found.**  "
        "Check `MAST_ROOT` and run the download script first."
    ))
else:
    rows = ""
    for key in sorted(groups):
        n   = len(groups[key])
        pct = 100 * n / len(spec_files)
        rows += f"<tr><td><code>{key}</code></td><td>{n}</td><td>{pct:.1f}%</td></tr>"

    display(Markdown(f"**Root:** `{_MAST_ROOT.resolve()}`"))
    display(Markdown(
        f"<table>"
        f"<thead><tr><th>Grating / Filter</th><th>Files</th><th>% of total</th></tr></thead>"
        f"<tbody>{rows}</tbody>"
        f"<tfoot><tr><td><strong>TOTAL</strong></td>"
        f"<td><strong>{len(spec_files)}</strong></td><td>100%</td></tr></tfoot>"
        f"</table>"
    ))


## 3b · Mascia+2024 catalog cross-match

Loads the official GLASS-JWST spectroscopic catalog (Mascia et al. 2024, A&A 690 A2)
and builds a lookup dict keyed on **MSA ID** extracted from each filename
(e.g. `100005` from `…_nirspec_abell2744_…-1324-100005_…_spec.fits`).

Each entry stores:
- `z` — spectroscopic redshift (`None` if not measured)
- `flag` — quality flag (4 = secure, 3 = reliable, 2 = fairly reliable, 1 = no z, 14 = continuum only)
- `ra`, `dec` — source coordinates
- `lines` — detected emission lines string (S/N > 5)

If `CATALOG_PATH` is `None` or the file is missing, `SOURCE_CATALOG` is empty
and the notebook falls back to FITS-header redshifts transparently.


In [ ]:
# ---------------------------------------------------------------------------
#  Download the Mascia+2024 catalog from MAST (run once)
# ---------------------------------------------------------------------------
# The catalog FITS is a small file (~200 KB) containing MSA IDs, coordinates,
# spectroscopic redshifts, quality flags, and detected emission lines for all
# 263 GLASS-JWST + DD-2756 spectra.
#
# Set FORCE_DOWNLOAD = True to re-download even if the file already exists.

FORCE_DOWNLOAD = False

_CATALOG_URI  = "mast:HLSP/glass-jwst/nirspec/hlsp_glass-jwst_jwst_nirspec_abell2744_v1_cat.fits"
_CATALOG_URL  = ("https://mast.stsci.edu/api/v0.1/Download/file"
                 "?uri=mast:HLSP/glass-jwst/nirspec/"
                 "hlsp_glass-jwst_jwst_nirspec_abell2744_v1_cat.fits")

# Destination: local directory next to the notebook (survives mastDownload cleanups)
_cat_dest = Path(CATALOG_PATH) if CATALOG_PATH else Path("./glass_jwst_cat.fits")

def _download_catalog(dest: Path, force: bool = False) -> bool:
    """
    Try three download methods in order:
      1. astroquery.mast  (preferred — authenticated, resumable)
      2. urllib fallback  (no extra dependencies)
    Returns True on success.
    """
    if dest.exists() and not force:
        size_kb = dest.stat().st_size // 1024
        display(Markdown(
            f"✅ Catalog already cached locally ({size_kb} KB): `{dest.resolve()}`  \n"
            "Set `FORCE_DOWNLOAD = True` to re-download."
        ))
        return True

    dest.parent.mkdir(parents=True, exist_ok=True)
    display(Markdown(f"⬇️ Downloading catalog to `{dest.resolve()}` …"))

    # ── Method 1: astroquery.mast ────────────────────────────────────────────
    try:
        from astroquery.mast import Observations
        # query_criteria with a provenance filter and product filename filter
        all_obs = Observations.query_criteria(provenance_name="glass-jwst")
        prods   = Observations.get_product_list(all_obs)
        mask    = ["v1_cat.fits" in str(r["productFilename"]) for r in prods]
        cat_prod = prods[mask]
        if len(cat_prod) > 0:
            manifest = Observations.download_products(
                cat_prod[:1],
                download_dir=str(dest.parent),   # download next to final destination
            )
            # Find the downloaded file and move/copy to expected path
            import shutil
            for row in manifest:
                dl_path = Path(row["Local Path"])
                if dl_path.exists() and dl_path.suffix == ".fits":
                    if dl_path.resolve() != dest.resolve():
                        shutil.copy2(dl_path, dest)
                    display(Markdown(f"✅ Downloaded via astroquery → `{dest.resolve()}`"))
                    return True
    except Exception as e:
        display(Markdown(f"ℹ️ astroquery method failed (`{e}`), trying direct URL …"))

    # ── Method 2: direct HTTPS download ──────────────────────────────────────
    try:
        import urllib.request, shutil
        with urllib.request.urlopen(_CATALOG_URL, timeout=60) as resp:
            if resp.status == 200:
                with open(dest, "wb") as fout:
                    shutil.copyfileobj(resp, fout)
                size_kb = dest.stat().st_size // 1024
                display(Markdown(
                    f"✅ Downloaded via HTTPS ({size_kb} KB) → `{dest.resolve()}`"
                ))
                return True
            else:
                display(Markdown(f"⚠️ HTTP {resp.status} — check URL or download manually."))
    except Exception as e:
        display(Markdown(f"⚠️ Direct download failed: `{e}`"))

    # ── Manual fallback instructions ──────────────────────────────────────────
    display(Markdown(
        "**Manual download:**  \n"
        "1. Go to https://archive.stsci.edu/hlsp/glass-jwst  \n"
        "2. Download `hlsp_glass-jwst_jwst_nirspec_abell2744_v1_cat.fits`  \n"
        f"3. Save it to: `{dest.resolve()}`  \n"
        "Then re-run this cell."
    ))
    return False


_download_ok = _download_catalog(_cat_dest, force=FORCE_DOWNLOAD)


In [ ]:
# ---------------------------------------------------------------------------
#  Flag descriptions — Mascia et al. 2024 Table 1
# ---------------------------------------------------------------------------
FLAG_DESCRIPTIONS = {
    4 : "Secure  (>99%, multiple lines)",
    3 : "Reliable  (single clear line)",
    2 : "Fairly reliable  (cross-corr + photo-z)",
    1 : "No reliable z  (no emission lines)",
    14: "Continuum only  (no emission lines)",
}

# ---------------------------------------------------------------------------
#  MSA-ID extraction from filename
# ---------------------------------------------------------------------------

def extract_msa_id(filepath) -> int | None:
    """
    Parse the MSA source ID from a GLASS-JWST NIRSpec filename.

    Convention (Mascia+2024):
      …_nirspec_abell2744_<grating>-<filter>_<program>-<msa_id>_v1_spec.fits

    e.g.  …_1324-100005_v1_spec.fits  →  100005
          …_2756-40202_v1_spec.fits   →  40202
    """
    stem   = Path(filepath).stem
    tokens = stem.split("_")
    for tok in reversed(tokens):
        if "-" in tok:
            for part in reversed(tok.split("-")):
                if part.isdigit() and len(part) >= 3:
                    return int(part)
        elif tok.isdigit() and len(tok) >= 3:
            return int(tok)
    return None


# ---------------------------------------------------------------------------
#  Catalog loader
# ---------------------------------------------------------------------------

def load_catalog(cat_path) -> dict:
    """
    Load the Mascia+2024 catalog FITS and return a dict keyed on MSA ID.

    The FITS file has two BinTableHDUs:
      HDU 1 — 152 rows, GLASS-JWST ERS (Program 1324)
      HDU 2 — 111 rows, JWST DD-2756   (Program 2756)
    Both are merged into a single dict (263 sources total).

    Emission line columns are 0/1 detection flags — we collect the names
    of all detected lines (value == 1) into a human-readable string.
    """
    # Non-science columns to skip when collecting emission lines
    _META_COLS = {"MSA_ID", "RA", "DEC", "PROGRAM", "Z_SPEC", "FLAG",
                  "ZSPEC", "REDSHIFT", "Z", "ZFLAG", "QUALITY"}

    if cat_path is None:
        display(Markdown("ℹ️ `CATALOG_PATH` is `None` — catalog lookup disabled."))
        return {}

    cat_path = Path(cat_path)
    if not cat_path.exists():
        msg = (
            f"⚠️ Catalog not found: `{cat_path}`  \n"
            "Download `hlsp_glass-jwst_jwst_nirspec_abell2744_v1_cat.fits` from MAST.  \n"
            "Falling back to FITS-header redshifts."
        )
        display(Markdown(msg))
        return {}

    def _pick(names, candidates):
        upper = {n.upper(): n for n in names}
        for c in candidates:
            if c.upper() in upper:
                return upper[c.upper()]
        return None

    catalog = {}
    try:
        with fits.open(cat_path) as hdul:
            # Collect ALL BinTableHDUs (HDU 1 = ERS, HDU 2 = DD-2756)
            tables = [h for h in hdul if isinstance(h, fits.BinTableHDU)]
            if not tables:
                display(Markdown("⚠️ No BinTableHDU found in catalog FITS."))
                return {}

            for tbl in tables:
                cols  = tbl.data.names
                c_id   = _pick(cols, ["MSA_ID", "ID", "MSAID", "SOURCE_ID"])
                c_z    = _pick(cols, ["Z_SPEC", "ZSPEC", "REDSHIFT", "Z"])
                c_flag = _pick(cols, ["FLAG", "ZFLAG", "QUALITY"])
                c_ra   = _pick(cols, ["RA", "RA_OBJ"])
                c_dec  = _pick(cols, ["DEC", "DEC_OBJ"])

                if c_id is None:
                    continue  # skip HDU if no ID column

                # Emission line columns: everything that isn't a meta column
                line_cols = [c for c in cols
                             if c.upper() not in _META_COLS
                             and c_id != c and c_z != c
                             and c_flag != c and c_ra != c and c_dec != c]

                for row in tbl.data:
                    try:
                        msa_id = int(row[c_id])
                    except (TypeError, ValueError):
                        continue

                    z_val = None
                    if c_z:
                        try:
                            zr = float(row[c_z])
                            if 0.0 <= zr < 25.0:
                                z_val = zr
                        except (TypeError, ValueError):
                            pass

                    flag_val = None
                    if c_flag:
                        try:
                            flag_val = int(row[c_flag])
                        except (TypeError, ValueError):
                            pass

                    # Build detected-lines string from 0/1 flag columns
                    detected = []
                    for lc in line_cols:
                        try:
                            if int(row[lc]) == 1:
                                detected.append(lc)
                        except (TypeError, ValueError):
                            pass
                    lines_str = ", ".join(detected)

                    # Later HDU wins on duplicate MSA_IDs (keeps DD-2756 entry)
                    catalog[msa_id] = {
                        "z"    : z_val,
                        "flag" : flag_val,
                        "ra"   : float(row[c_ra])  if c_ra  else None,
                        "dec"  : float(row[c_dec]) if c_dec else None,
                        "lines": lines_str,
                    }

    except Exception as exc:
        display(Markdown(f"⚠️ Error reading catalog: `{exc}`"))
        return {}

    return catalog


# ── Load and report ───────────────────────────────────────────────────────────
_cat_path      = Path(CATALOG_PATH) if CATALOG_PATH is not None else None
SOURCE_CATALOG = load_catalog(_cat_path)

if SOURCE_CATALOG:
    n_with_z    = sum(1 for v in SOURCE_CATALOG.values() if v["z"] is not None)
    flag_counts: dict = {}
    for v in SOURCE_CATALOG.values():
        f = v["flag"]
        flag_counts[f] = flag_counts.get(f, 0) + 1

    rows = "".join(
        f"<tr><td><b>{f}</b></td>"
        f"<td>{FLAG_DESCRIPTIONS.get(f, '?')}</td>"
        f"<td>{flag_counts[f]}</td></tr>"
        for f in sorted(flag_counts, key=lambda x: (x is None, x))
    )
    display(Markdown(
        f"**Catalog loaded ✓** — {len(SOURCE_CATALOG)} sources "
        f"({n_with_z} with spectroscopic z)  \n"
        f"Path: `{_cat_path.resolve()}`"
    ))
    display(Markdown(
        "<table><thead><tr><th>Flag</th><th>Meaning</th><th>Count</th></tr></thead>"
        f"<tbody>{rows}</tbody></table>"
    ))
else:
    SOURCE_CATALOG = {}
    display(Markdown("ℹ️ `SOURCE_CATALOG` is empty — using FITS-header redshifts only."))


In [ ]:
# ---------------------------------------------------------------------------
#  Catalog + filename diagnostic
#  Run this cell to understand why auto-mode may be falling through to z_quad
# ---------------------------------------------------------------------------

print("=" * 65)
print("CATALOG DIAGNOSTIC")
print("=" * 65)

# ── 1. Catalog state ─────────────────────────────────────────────────────────
print(f"\nSOURCE_CATALOG entries : {len(SOURCE_CATALOG)}")
if SOURCE_CATALOG:
    sample_keys = list(SOURCE_CATALOG.keys())[:5]
    print(f"Sample MSA IDs in catalog : {sample_keys}")
    n_with_z    = sum(1 for v in SOURCE_CATALOG.values() if v["z"] is not None)
    n_flag1     = sum(1 for v in SOURCE_CATALOG.values() if v["flag"] in (1, 14))
    print(f"Entries with confirmed z  : {n_with_z}")
    print(f"Entries with flag 1 or 14 : {n_flag1}  (no z by design)")
    print(f"Sample entries:")
    for k in sample_keys:
        print(f"  {k}: {SOURCE_CATALOG[k]}")
else:
    print("  ⚠️  SOURCE_CATALOG is EMPTY — catalog was not loaded.")
    print(f"  CATALOG_PATH = {CATALOG_PATH!r}")
    import os
    if CATALOG_PATH:
        print(f"  File exists  : {os.path.exists(CATALOG_PATH)}")

# ── 2. Raw catalog columns (re-open to inspect ALL HDUs) ─────────────────────
if CATALOG_PATH:
    from astropy.io import fits as _fits
    from pathlib import Path as _Path
    _cp = _Path(CATALOG_PATH)
    if _cp.exists():
        with _fits.open(_cp) as hdul:
            print(f"\nCatalog HDUs:")
            for i, h in enumerate(hdul):
                data = h.data
                if data is None:
                    shape_str = "None"
                elif hasattr(data, "names"):          # BinTable
                    shape_str = f"{len(data)} rows × {len(data.names)} cols"
                else:
                    shape_str = str(data.shape)
                print(f"  [{i}] {type(h).__name__:20s}  {shape_str}")

            # Check every HDU for table-like data
            for i, h in enumerate(hdul):
                if isinstance(h, (_fits.BinTableHDU, _fits.TableHDU)):
                    print(f"\nTable HDU [{i}] columns : {list(h.data.names)}")
                    print(f"Table HDU [{i}] rows    : {len(h.data)}")
                    print(f"First 3 rows:")
                    for row in h.data[:3]:
                        print(f"  { {c: str(row[c])[:30] for c in h.data.names} }")
    else:
        print(f"\n⚠️  Catalog file does not exist: {_cp}")

# ── 3. Filename → MSA ID cross-match for first 10 spec files ─────────────────
print(f"\n{'File stem':<52}  MSA ID    In cat?  z")
print("-" * 82)
for fpath in spec_files[:10]:
    msa_id = extract_msa_id(fpath)
    entry  = SOURCE_CATALOG.get(msa_id) if msa_id is not None else None
    z_val  = entry["z"] if entry else None
    flag   = entry["flag"] if entry else None
    in_cat = f"✓ f{flag}" if entry else "✗"
    z_str  = f"{z_val:.4f}" if z_val is not None else "—"
    stem   = fpath.stem[-52:] if len(fpath.stem) > 52 else fpath.stem
    print(f"  {stem:<52}  {str(msa_id):<9} {in_cat:<8} {z_str}")

# ── 4. Auto-mode preview: how many would get single vs z_quad ────────────────
print(f"\n{'AUTO-MODE PREVIEW':=^65}")
known   = [f for f in spec_files[IDX_START:IDX_END+1]
           if SOURCE_CATALOG.get(extract_msa_id(f), {}).get("z") is not None]
unknown = [f for f in spec_files[IDX_START:IDX_END+1]
           if SOURCE_CATALOG.get(extract_msa_id(f), {}).get("z") is None]
print(f"Range IDX {IDX_START}–{IDX_END}  ({IDX_END-IDX_START+1} spectra)")
print(f"  Single panel (catalog z known) : {len(known)}")
print(f"  z_quad (catalog z unknown)     : {len(unknown)}")
if unknown:
    print(f"  Unknown-z MSA IDs: "
          + ", ".join(str(extract_msa_id(f)) for f in unknown))
print("=" * 65)


## 4 · Metadata & spectrum loading helpers

In [ ]:
_META_KEYS = [
    ("OBJECT",   "Object",    "{}"),
    ("TARGNAME", "Target",    "{}"),
    ("RA_TARG",  "RA",        "{:.5f}°"),
    ("DEC_TARG", "Dec",       "{:.5f}°"),
    ("SRCRA",    "Src RA",    "{:.5f}°"),
    ("SRCDEC",   "Src Dec",   "{:.5f}°"),
    ("REDSHIFT", "z",         "{:.4f}"),
    ("Z_SPEC",   "z_spec",    "{:.4f}"),
    ("Z_PHOT",   "z_phot",    "{:.4f}"),
    ("INSTRUME", "Instr",     "{}"),
    ("GRATING",  "Grating",   "{}"),
    ("FILTER",   "Filter",    "{}"),
    ("DISPERSR", "Disperser", "{}"),
    ("EXPTIME",  "ExpTime",   "{:.0f}s"),
    ("SRCTYPE",  "SrcType",   "{}"),
    ("SLITID",   "SlitID",    "{}"),
    ("SOURCEID", "SourceID",  "{}"),
]


def build_title(hdr: fits.Header, filepath: Path) -> tuple[str, str]:
    """Build panel title, merging FITS header with Mascia+2024 catalog."""
    tokens = filepath.stem.replace("-", "_").split("_")
    src_id = next((t for t in reversed(tokens) if t.isdigit()), filepath.stem)
    grism  = next(
        (t for t in tokens if any(g in t for g in ["g140h", "g235h", "g395h", "prism"])),
        "?",
    )
    # FITS header fields
    meta = {}
    for kw, label, fmt in _META_KEYS:
        val = hdr.get(kw, None)
        if val is not None and str(val).strip() not in ("", "N/A", "UNKNOWN"):
            try:
                meta[label] = fmt.format(val)
            except (ValueError, TypeError):
                meta[label] = str(val).strip()

    instr_str = meta.get("Instr", "NIRSpec")
    grat_str  = meta.get("Grating") or meta.get("Disperser") or grism.upper()
    filt_str  = meta.get("Filter", "")
    exp_str   = meta.get("ExpTime", "")

    # Catalog lookup (Mascia+2024) — wins over header for z and coords
    msa_id    = extract_msa_id(filepath)
    cat_entry = SOURCE_CATALOG.get(msa_id) if msa_id is not None else None

    if cat_entry is not None:
        z_val    = cat_entry["z"]
        z_str    = f"{z_val:.4f}" if z_val is not None else "—"
        flag_val = cat_entry["flag"]
        flag_str = (f"Flag {flag_val} [{FLAG_DESCRIPTIONS.get(flag_val, "?")}]"
                    if flag_val is not None else "")
        ra_val   = cat_entry["ra"]
        dec_val  = cat_entry["dec"]
        ra_str   = f"{ra_val:.5f}°"  if ra_val  is not None else ""
        dec_str  = f"{dec_val:.5f}°" if dec_val is not None else ""
        lines_str = cat_entry.get("lines", "")
        msa_str   = f"MSA {msa_id}"
    else:
        z_str     = meta.get("z") or meta.get("z_spec") or meta.get("z_phot") or "—"
        flag_str  = ""
        ra_str    = meta.get("Src RA") or meta.get("RA", "")
        dec_str   = meta.get("Src Dec") or meta.get("Dec", "")
        lines_str = ""
        msa_str   = f"Source {src_id}"

    suptitle = (
        f"{msa_str}  |  {instr_str} / {grat_str}"
        + (f" + {filt_str}" if filt_str else "")
        + f"  |  z = {z_str}"
        + (f"  |  {flag_str}" if flag_str else "")
    )
    parts = []
    if ra_str and dec_str:
        parts.append(f"(RA, Dec) = ({ra_str}, {dec_str})")
    if exp_str:
        parts.append(f"Exp = {exp_str}")
    if lines_str:
        disp = lines_str if len(lines_str) <= 80 else lines_str[:77] + "…"
        parts.append(f"Lines: {disp}")
    parts.append(f"File: {filepath.name}")
    return suptitle, "   ·   ".join(parts)


def _find_col(table, name_variants):
    cols = {c.upper() for c in table.names}
    for v in name_variants:
        if v.upper() in cols:
            return table[v]
    return None


def load_1d(hdul):
    for hdu in hdul[1:]:
        if not isinstance(hdu, fits.BinTableHDU):
            continue
        wave = _find_col(hdu.data, _WAVE_NAMES)
        flux = _find_col(hdu.data, _FLUX_NAMES)
        if wave is None or flux is None:
            continue
        err = _find_col(hdu.data, _ERR_NAMES)
        w   = np.asarray(wave, dtype=float).ravel()
        f   = np.asarray(flux, dtype=float).ravel()
        e   = np.asarray(err,  dtype=float).ravel() if err is not None else None
        if np.nanmedian(w) < 50:
            w = w * 1e4
        return w, f, e
    return None


def load_2d(hdul):
    for hdu in hdul[1:]:
        if isinstance(hdu, (fits.ImageHDU, fits.PrimaryHDU)):
            d = hdu.data
            if d is not None and d.ndim == 2 and max(d.shape) > 10:
                return d.astype(float)
    return None


def _error_panel(ax, fpath, msg, idx):
    ax.set_facecolor("#1a0d0d")
    ax.text(0.5, 0.5, f"[{idx}] {fpath.name}\n{msg}",
            transform=ax.transAxes, ha="center", va="center",
            color="#ff6666", fontsize=8, wrap=True)
    ax.set_xticks([])
    ax.set_yticks([])


print("Helpers defined ✓")


## 5 · Plot spectra

**`range` mode** — plots `IDX_START` → `IDX_END` with one panel per spectrum.  
**`z_quad` mode** — plots one 2×2 figure per spectrum, each quadrant showing the
same flux with a different redshift applied to the emission line annotations.
The quadrant where annotated lines align with real flux peaks reveals the
true redshift.

Re-run just this cell after changing any parameter in Cell 1.


In [ ]:
# ---------------------------------------------------------------------------
#  Shared legend helper
# ---------------------------------------------------------------------------

def _make_line_legend(fig, active_groups: set) -> None:
    """Add a compact colour-coded legend for the emission line groups."""
    handles = []
    for grp_key in ["hydrogen", "forbidden", "agn_uv", "agn_coronal", "sfr"]:
        if grp_key not in active_groups:
            continue
        grp = EMISSION_LINES[grp_key]
        handles.append(
            plt.Line2D([0], [0],
                       color=grp["color"], linewidth=1.2, linestyle="--",
                       label=grp_key.replace("_", " ").title())
        )
    if handles:
        fig.legend(
            handles=handles,
            loc="lower center",
            ncol=len(handles),
            fontsize=7,
            framealpha=0.25,
            facecolor="#1a1a2e",
            edgecolor="#444466",
            labelcolor="white",
            bbox_to_anchor=(0.5, 0.0),
        )


# ---------------------------------------------------------------------------
#  Helper: draw one 1-D spectrum panel
# ---------------------------------------------------------------------------

def _draw_1d_panel(
    ax,
    w: np.ndarray,
    f_arr: np.ndarray,
    e,                        # ndarray | None
    redshift: float | None,
    z_label: str,
    global_idx: int,
    fpath,
    suptitle: str,
    subtitle: str,
    col: int,
    active_groups: set,
    show_z_source: str = "",  # extra label suffix, e.g. " (override)"
) -> None:
    """Render a single 1-D spectrum with optional emission line annotation."""
    ax.set_facecolor("#0d0d1a")
    for spine in ax.spines.values():
        spine.set_edgecolor("#2a2a4a")

    fmed = np.nanmedian(f_arr)
    fsig = np.nanstd(f_arr)
    ylo  = fmed - 2.0 * fsig
    yhi  = fmed + 5.0 * fsig

    if e is not None:
        ax.fill_between(w, f_arr - e, f_arr + e,
                        color="#4466aa", alpha=0.35, linewidth=0)

    ax.plot(w, f_arr, color="#88ccff", linewidth=0.7, alpha=0.9)
    ax.axhline(0, color="#555577", linewidth=0.6, linestyle="--")
    ax.set_xlim(w.min(), w.max())
    ax.set_ylim(ylo, yhi)

    if active_groups:
        annotate_emission_lines(
            ax=ax, wave_obs=w, ylo=ylo, yhi=yhi,
            redshift=redshift, groups=active_groups,
        )

    # z label — top-right corner, colour signals whether lines were drawn
    if redshift is not None:
        ax.text(0.98, 0.96, f"z = {redshift:.4f}{show_z_source}",
                transform=ax.transAxes, ha="right", va="top",
                color="#ffcc88", fontsize=7, fontweight="bold", zorder=6)
    else:
        ax.text(0.98, 0.96, "z unknown",
                transform=ax.transAxes, ha="right", va="top",
                color="#886644", fontsize=6.5, style="italic", zorder=6)

    if col == 0:
        ax.set_ylabel("Flux (μJy)", color="#aaaacc", fontsize=8)
    ax.set_xlabel("Wavelength (Å)", color="#aaaacc", fontsize=8)
    ax.tick_params(colors="#aaaacc", labelsize=7)

    # Panel title — in z_quad mode we show the window label; in range mode full metadata
    if z_label:
        ax.set_title(z_label, color="#ffcc88", fontsize=8,
                     fontweight="bold", pad=3, loc="left")
    else:
        ax.set_title(
            f"[{global_idx}]  {suptitle}\n"
            f"$\\it{{{subtitle.replace(chr(95), chr(95))}}}$",
            color="white", fontsize=7.5, pad=4, loc="left",
        )


# ---------------------------------------------------------------------------
#  Mode A: range plot  (original one-panel-per-spectrum layout)
# ---------------------------------------------------------------------------

def plot_range(
    files, idx_start, idx_end,
    show_2d=False, show_lines=True, line_groups=None, save=True,
):
    """One panel per spectrum across IDX_START–IDX_END."""
    idx_start = max(0, idx_start)
    idx_end   = min(idx_end, len(files) - 1)
    if idx_start > idx_end:
        display(Markdown(f"⚠️ Nothing to plot: IDX_START ({idx_start}) > IDX_END ({idx_end})."))
        return

    active_groups = (line_groups or set()) if show_lines else set()
    subset = files[idx_start : idx_end + 1]
    display(Markdown(
        f"**[range] Plotting {idx_start}–{idx_end}** "
        f"({len(subset)} panels · lines: "
        f"{', '.join(sorted(active_groups)) if active_groups else 'off'})"
    ))

    n_rows     = (len(subset) + COLS_PER_ROW - 1) // COLS_PER_ROW
    row_h      = PANEL_HEIGHT * (2 if show_2d else 1)
    bottom_pad = 0.06 if active_groups else 0.04
    fig_h      = row_h * n_rows + 1.2

    fig = plt.figure(figsize=(FIG_WIDTH, fig_h), facecolor="#0d0d1a")
    fig.suptitle(
        f"GLASS-JWST NIRSpec Spectra — Abell 2744\n"
        f"Indices {idx_start}–{idx_end}  of  {len(files)} spectra",
        color="white", fontsize=13, fontweight="bold", y=0.998,
    )
    outer = gridspec.GridSpec(
        n_rows, COLS_PER_ROW, figure=fig,
        hspace=0.62, wspace=0.32,
        top=0.96, bottom=bottom_pad, left=0.06, right=0.97,
    )

    for panel_idx, fpath in enumerate(subset):
        row        = panel_idx // COLS_PER_ROW
        col        = panel_idx %  COLS_PER_ROW
        global_idx = idx_start + panel_idx

        if show_2d:
            inner = gridspec.GridSpecFromSubplotSpec(
                2, 1, subplot_spec=outer[row, col],
                height_ratios=[1, 2.5], hspace=0.08,
            )
            ax2d = fig.add_subplot(inner[0])
            ax1d = fig.add_subplot(inner[1])
        else:
            ax1d = fig.add_subplot(outer[row, col])
            ax2d = None

        try:
            with fits.open(fpath, memmap=False) as hdul:
                primary_hdr = hdul[0].header
                combined    = primary_hdr.copy()
                if len(hdul) > 1 and hasattr(hdul[1], "header"):
                    for card in hdul[1].header.cards:
                        if card.keyword not in combined:
                            combined.append(card)
                result_1d = load_1d(hdul)
                img_2d    = load_2d(hdul) if show_2d else None
        except Exception as exc:
            _error_panel(ax1d, fpath, str(exc), global_idx)
            if ax2d:
                ax2d.set_visible(False)
            continue

        suptitle, subtitle = build_title(combined, fpath)
        redshift = extract_redshift(combined, fpath)

        if ax2d is not None and img_2d is not None:
            vmed = np.nanmedian(img_2d); vsig = np.nanstd(img_2d)
            ax2d.imshow(img_2d, origin="lower", aspect="auto", cmap="inferno",
                        norm=AsinhNorm(linear_width=max(vsig*0.5,1e-30),
                                       vmin=vmed-vsig, vmax=vmed+8*vsig),
                        interpolation="nearest")
            ax2d.set_xticks([])
            ax2d.set_ylabel("Spatial", color="#aaaacc", fontsize=7)
            ax2d.tick_params(colors="#aaaacc", labelsize=6)
            for spine in ax2d.spines.values():
                spine.set_edgecolor("#333355")
            ax2d.set_facecolor("#0d0d1a")
        elif ax2d is not None:
            ax2d.set_visible(False)

        if result_1d is None:
            _error_panel(ax1d, fpath, "No 1-D spectrum found", global_idx)
            continue

        wave, flux, err = result_1d
        good = np.isfinite(wave) & np.isfinite(flux)
        if err is not None:
            good &= np.isfinite(err) & (err > 0)
        if good.sum() < 3:
            _error_panel(ax1d, fpath, "Insufficient finite pixels", global_idx)
            continue

        w, f_arr = wave[good], flux[good]
        e        = err[good] if err is not None else None

        z_src = ""
        try:
            if Z_OVERRIDE is not None and redshift is not None:
                if abs(float(Z_OVERRIDE) - redshift) < 1e-6:
                    z_src = " (override)"
        except (NameError, TypeError):
            pass

        _draw_1d_panel(
            ax=ax1d, w=w, f_arr=f_arr, e=e,
            redshift=redshift, z_label="",
            global_idx=global_idx, fpath=fpath,
            suptitle=suptitle, subtitle=subtitle,
            col=col, active_groups=active_groups,
            show_z_source=z_src,
        )

    if active_groups:
        _make_line_legend(fig, active_groups)
    plt.tight_layout(rect=[0, bottom_pad, 1, 0.97])

    if save:
        _OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        out = _OUTPUT_DIR / f"glass_jwst_spectra_{idx_start:04d}_{idx_end:04d}.png"
        fig.savefig(out, dpi=150, bbox_inches="tight",
                    facecolor="#0d0d1a", pad_inches=0.08)
        display(Markdown(f"💾 Saved → `{out.resolve()}`"))
    plt.show()


# ---------------------------------------------------------------------------
#  Mode B: z_quad plot  (2×2 per target, four redshift windows)
# ---------------------------------------------------------------------------

def plot_z_quad(
    files, idx_start, idx_end,
    z_windows, show_lines=True, line_groups=None, save=True,
):
    """
    For each spectrum in idx_start–idx_end, produce one figure with a 2×2
    grid of panels — same flux data, four different redshift annotations.
    Each quadrant applies a different z from z_windows so you can visually
    identify which redshift makes the annotated lines land on flux peaks.
    """
    idx_start = max(0, idx_start)
    idx_end   = min(idx_end, len(files) - 1)
    if idx_start > idx_end:
        display(Markdown(f"⚠️ Nothing to plot."))
        return

    active_groups = (line_groups or set()) if show_lines else set()
    n_targets = idx_end - idx_start + 1
    display(Markdown(
        f"**[z_quad] Plotting {n_targets} target(s), indices {idx_start}–{idx_end}** — "
        f"one 2×2 figure per target · "
        f"z-windows: {[lbl for lbl,_ in z_windows]} · "
        f"lines: {', '.join(sorted(active_groups)) if active_groups else 'off'}"
    ))

    for spec_idx in range(idx_start, idx_end + 1):
        fpath      = files[spec_idx]
        global_idx = spec_idx

        # Load spectrum once
        try:
            with fits.open(fpath, memmap=False) as hdul:
                primary_hdr = hdul[0].header
                combined    = primary_hdr.copy()
                if len(hdul) > 1 and hasattr(hdul[1], "header"):
                    for card in hdul[1].header.cards:
                        if card.keyword not in combined:
                            combined.append(card)
                result_1d = load_1d(hdul)
        except Exception as exc:
            display(Markdown(f"⚠️ [{global_idx}] Could not load `{fpath.name}`: {exc}"))
            continue

        if result_1d is None:
            display(Markdown(f"⚠️ [{global_idx}] No 1-D spectrum in `{fpath.name}` — skipped."))
            continue

        wave, flux, err = result_1d
        good = np.isfinite(wave) & np.isfinite(flux)
        if err is not None:
            good &= np.isfinite(err) & (err > 0)
        if good.sum() < 3:
            display(Markdown(f"⚠️ [{global_idx}] Too few finite pixels — skipped."))
            continue

        w, f_arr = wave[good], flux[good]
        e        = err[good] if err is not None else None

        suptitle, subtitle = build_title(combined, fpath)
        # Catalog z takes priority for the z_quad super-title
        _msa  = extract_msa_id(fpath)
        hdr_z = (SOURCE_CATALOG.get(_msa, {}).get('z')
                 if SOURCE_CATALOG and _msa else None)
        if hdr_z is None:
            hdr_z = extract_redshift_from_header(combined)

        # ── Build figure ──────────────────────────────────────────────────
        fig, axes = plt.subplots(
            2, 2,
            figsize=(FIG_WIDTH, PANEL_HEIGHT * 2.1),
            facecolor="#0d0d1a",
        )
        fig.patch.set_facecolor("#0d0d1a")

        # Super-title: source metadata
        z_hdr_str = f"  |  header z = {hdr_z:.4f}" if hdr_z is not None else ""
        fig.suptitle(
            f"[{global_idx}]  {suptitle}{z_hdr_str}\n"
            f"$\\it{{{subtitle.replace(chr(95), chr(95))}}}$\n"
            f"Four redshift windows — annotated lines shift with z",
            color="white", fontsize=9, fontweight="bold",
            y=0.995, va="top",
        )

        for quad_idx, (z_label, z_val) in enumerate(z_windows):
            ax  = axes[quad_idx // 2][quad_idx % 2]
            col = quad_idx % 2

            _draw_1d_panel(
                ax=ax, w=w, f_arr=f_arr, e=e,
                redshift=z_val,
                z_label=z_label,
                global_idx=global_idx, fpath=fpath,
                suptitle=suptitle, subtitle=subtitle,
                col=col, active_groups=active_groups,
                show_z_source="",
            )

        if active_groups:
            _make_line_legend(fig, active_groups)

        plt.tight_layout(rect=[0, 0.06 if active_groups else 0.02, 1, 0.93])

        if save:
            _OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
            out = _OUTPUT_DIR / f"glass_jwst_zquad_{global_idx:04d}.png"
            fig.savefig(out, dpi=150, bbox_inches="tight",
                        facecolor="#0d0d1a", pad_inches=0.08)
            display(Markdown(f"💾 [{global_idx}] Saved → `{out.resolve()}`"))
        plt.show()


def extract_redshift_from_header(hdr) -> float | None:
    """Header-only redshift (no Z_OVERRIDE), used for informational display in z_quad."""
    for kw in ("REDSHIFT", "Z_SPEC", "Z_PHOT", "ZNEW", "Z"):
        val = hdr.get(kw, None)
        if val is not None:
            try:
                z = float(val)
                if 0.0 <= z < 25.0:
                    return z
            except (TypeError, ValueError):
                pass
    return None



# ---------------------------------------------------------------------------
#  Grating grouping helpers
# ---------------------------------------------------------------------------

_GRATING_ORDER = ["f100lp-g140h", "f170lp-g235h", "f290lp-g395h",
                  "clear-prism", "prism"]

def _grating_key(fpath) -> str:
    stem = Path(fpath).stem.lower()
    for g in _GRATING_ORDER:
        if g in stem:
            return g
    for tok in stem.split("_"):
        if any(x in tok for x in ["g140h", "g235h", "g395h", "prism"]):
            return tok
    return stem


def group_by_target(files: list) -> dict:
    """
    Group spec files by MSA ID so all gratings for one source plot together.
    Returns {msa_id: [fpath, ...]} sorted short->long wavelength per group.
    """
    from collections import defaultdict
    groups = defaultdict(list)
    order  = []
    for f in files:
        mid = extract_msa_id(f)
        key = mid if mid is not None else str(f)
        if key not in groups:
            order.append(key)
        groups[key].append(f)
    for key in order:
        groups[key].sort(key=lambda f: (
            _GRATING_ORDER.index(_grating_key(f))
            if _grating_key(f) in _GRATING_ORDER else 99
        ))
    return {k: groups[k] for k in order}


# ---------------------------------------------------------------------------
#  Shared spectrum loader
# ---------------------------------------------------------------------------

def _load_spectrum(fpath):
    """Return (combined_hdr, result_1d). Both None on error."""
    try:
        with fits.open(fpath, memmap=False) as hdul:
            combined = hdul[0].header.copy()
            if len(hdul) > 1 and hasattr(hdul[1], "header"):
                for card in hdul[1].header.cards:
                    if card.keyword not in combined:
                        combined.append(card)
            result_1d = load_1d(hdul)
        return combined, result_1d
    except Exception:
        return None, None


# ---------------------------------------------------------------------------
#  Per-target figure: stacked grating panels
# ---------------------------------------------------------------------------

def _plot_target_figure(msa_id, grating_files, global_idx,
                        active_groups, redshift, flag_val, save, mode_label):
    """
    One figure per MSA target: stacked panels, one per grating file,
    ordered short->long wavelength. Source metadata in the suptitle.
    """
    n_panels = len(grating_files)
    bot_pad  = 0.07 if active_groups else 0.03
    fig, axes = plt.subplots(
        n_panels, 1,
        figsize=(FIG_WIDTH, PANEL_HEIGHT * n_panels + 1.4),
        facecolor="#0d0d1a", squeeze=False,
    )
    fig.patch.set_facecolor("#0d0d1a")

    # ── Suptitle: source-level metadata ──────────────────────────────────────
    cat_entry  = SOURCE_CATALOG.get(msa_id, {})
    flag_desc  = FLAG_DESCRIPTIONS.get(flag_val, "")
    ra_val     = cat_entry.get("ra")
    dec_val    = cat_entry.get("dec")
    coord_str  = (f"(RA, Dec) = ({ra_val:.5f}\u00b0, {dec_val:.5f}\u00b0)"
                  if ra_val is not None else "")
    lines_str  = cat_entry.get("lines", "")
    lines_disp = (lines_str[:90] + "\u2026") if len(lines_str) > 90 else lines_str
    z_str      = f"{redshift:.4f}" if redshift is not None else "unknown"
    flag_str   = f"Flag {flag_val} [{flag_desc}]" if flag_val is not None else ""

    line1 = ("MSA " + str(msa_id) + "  |  NIRSpec  |  z = " + z_str
             + ("  |  " + flag_str if flag_str else ""))
    parts = [p for p in [coord_str,
                          ("Lines: " + lines_disp) if lines_disp else ""] if p]
    line2 = "   \u00b7   ".join(parts)

    fig.suptitle(
        line1 + ("\n" + line2 if line2 else ""),
        color="white", fontsize=9, fontweight="bold", y=0.998, va="top",
    )

    # ── One panel per grating ─────────────────────────────────────────────────
    for panel_idx, fpath in enumerate(grating_files):
        ax      = axes[panel_idx][0]
        grating = _grating_key(fpath).upper()

        combined, result_1d = _load_spectrum(fpath)
        if result_1d is None:
            _error_panel(ax, fpath, "No 1-D spectrum", panel_idx)
            continue

        wave, flux, err = result_1d
        good = np.isfinite(wave) & np.isfinite(flux)
        if err is not None:
            good &= np.isfinite(err) & (err > 0)
        if good.sum() < 3:
            _error_panel(ax, fpath, "Insufficient finite pixels", panel_idx)
            continue

        w, f_arr = wave[good], flux[good]
        e        = err[good] if err is not None else None

        _draw_1d_panel(
            ax=ax, w=w, f_arr=f_arr, e=e,
            redshift=redshift, z_label="",
            global_idx=global_idx, fpath=fpath,
            suptitle="", subtitle="",
            col=0, active_groups=active_groups,
            show_z_source="",
        )
        ax.set_title(grating, color="#aaddff", fontsize=8,
                     fontweight="bold", pad=3, loc="left")

    if active_groups:
        _make_line_legend(fig, active_groups)
    plt.tight_layout(rect=[0, bot_pad, 1, 0.96])

    if save:
        _OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        out = _OUTPUT_DIR / ("glass_jwst_" + mode_label + "_" + str(msa_id) + ".png")
        fig.savefig(out, dpi=150, bbox_inches="tight",
                    facecolor="#0d0d1a", pad_inches=0.08)
        display(Markdown("💾 MSA " + str(msa_id) + " saved → `" + str(out.resolve()) + "`"))
    plt.show()


# ---------------------------------------------------------------------------
#  _has_catalog_z
# ---------------------------------------------------------------------------

def _has_catalog_z(fpath) -> bool:
    if not SOURCE_CATALOG:
        return False
    msa_id = extract_msa_id(fpath)
    if msa_id is None:
        return False
    entry = SOURCE_CATALOG.get(msa_id)
    return entry is not None and entry["z"] is not None


# ---------------------------------------------------------------------------
#  Mode A: range
# ---------------------------------------------------------------------------

def plot_range(files, idx_start, idx_end,
               show_2d=False, show_lines=True, line_groups=None, save=True):
    """One figure per MSA target, stacked grating panels, single z."""
    idx_start     = max(0, idx_start)
    idx_end       = min(idx_end, len(files) - 1)
    subset        = files[idx_start : idx_end + 1]
    target_groups = group_by_target(subset)
    active_groups = (line_groups or set()) if show_lines else set()

    display(Markdown(
        "**[range] " + str(len(target_groups)) +
        " unique targets, indices " + str(idx_start) + "\u2013" + str(idx_end) + "**"
    ))
    for msa_id, grating_files in target_groups.items():
        redshift = extract_redshift({}, grating_files[0])
        flag_val = SOURCE_CATALOG.get(msa_id, {}).get("flag")
        _plot_target_figure(
            msa_id=msa_id, grating_files=grating_files,
            global_idx=files.index(grating_files[0]),
            active_groups=active_groups,
            redshift=redshift, flag_val=flag_val,
            save=save, mode_label="range",
        )


# ---------------------------------------------------------------------------
#  Mode B: z_quad
# ---------------------------------------------------------------------------

def plot_z_quad(files, idx_start, idx_end,
                z_windows, show_lines=True, line_groups=None, save=True):
    """2x2 z-window grid — uses first grating file per target."""
    idx_start     = max(0, idx_start)
    idx_end       = min(idx_end, len(files) - 1)
    subset        = files[idx_start : idx_end + 1]
    target_groups = group_by_target(subset)
    active_groups = (line_groups or set()) if show_lines else set()

    display(Markdown(
        "**[z_quad] " + str(len(target_groups)) + " targets**"
    ))
    for msa_id, grating_files in target_groups.items():
        fpath      = grating_files[0]
        global_idx = files.index(fpath)
        combined, result_1d = _load_spectrum(fpath)
        if result_1d is None:
            display(Markdown("⚠️ MSA " + str(msa_id) + ": no spectrum — skipped."))
            continue
        wave, flux, err = result_1d
        good = np.isfinite(wave) & np.isfinite(flux)
        if err is not None:
            good &= np.isfinite(err) & (err > 0)
        if good.sum() < 3:
            continue
        w, f_arr  = wave[good], flux[good]
        e         = err[good] if err is not None else None
        suptitle, subtitle = build_title(combined, fpath)
        hdr_z = extract_redshift_from_header(combined)

        fig, axes_2x2 = plt.subplots(
            2, 2, figsize=(FIG_WIDTH, PANEL_HEIGHT * 2.1), facecolor="#0d0d1a"
        )
        fig.patch.set_facecolor("#0d0d1a")
        z_hdr_str = ("  |  header z = " + f"{hdr_z:.4f}") if hdr_z is not None else ""
        fig.suptitle(
            "MSA " + str(msa_id) + "  |  " + suptitle + z_hdr_str + "\n"
            "z unknown \u2014 four redshift windows for identification",
            color="white", fontsize=9, fontweight="bold", y=0.995, va="top",
        )
        for quad_idx, (z_label, z_val) in enumerate(z_windows):
            ax = axes_2x2[quad_idx // 2][quad_idx % 2]
            _draw_1d_panel(
                ax=ax, w=w, f_arr=f_arr, e=e,
                redshift=z_val, z_label=z_label,
                global_idx=global_idx, fpath=fpath,
                suptitle=suptitle, subtitle=subtitle,
                col=quad_idx % 2, active_groups=active_groups,
            )
        if active_groups:
            _make_line_legend(fig, active_groups)
        plt.tight_layout(rect=[0, 0.06 if active_groups else 0.02, 1, 0.93])
        if save:
            _OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
            out = _OUTPUT_DIR / ("glass_jwst_zquad_" + str(msa_id) + ".png")
            fig.savefig(out, dpi=150, bbox_inches="tight",
                        facecolor="#0d0d1a", pad_inches=0.08)
            display(Markdown("💾 MSA " + str(msa_id) + " z_quad saved → `" + str(out.resolve()) + "`"))
        plt.show()


# ---------------------------------------------------------------------------
#  Mode C: auto
# ---------------------------------------------------------------------------

def _run_auto(files, idx_start, idx_end, show_lines, line_groups, save):
    """
    Auto mode: group by MSA target, then route each target:
      catalog z known  -> stacked grating panels at confirmed z
      catalog z unknown -> 2x2 z_quad grid (first grating only)
    """
    subset        = files[idx_start : idx_end + 1]
    target_groups = group_by_target(subset)
    active_groups = (line_groups or set()) if show_lines else set()

    known_ids   = [mid for mid, fps in target_groups.items()
                   if _has_catalog_z(fps[0])]
    unknown_ids = [mid for mid, fps in target_groups.items()
                   if not _has_catalog_z(fps[0])]

    display(Markdown(
        "**[auto] " + str(len(target_groups)) + " unique targets, "
        "indices " + str(idx_start) + "\u2013" + str(idx_end) + "** \u2014 "
        + str(len(known_ids)) + " with catalog z (stacked grating panels) \u00b7 "
        + str(len(unknown_ids)) + " without (z_quad)"
    ))

    for msa_id in known_ids:
        grating_files = target_groups[msa_id]
        cat_entry     = SOURCE_CATALOG.get(msa_id, {})
        _plot_target_figure(
            msa_id=msa_id, grating_files=grating_files,
            global_idx=files.index(grating_files[0]),
            active_groups=active_groups,
            redshift=cat_entry.get("z"),
            flag_val=cat_entry.get("flag"),
            save=save, mode_label="auto",
        )

    if unknown_ids:
        display(Markdown(
            "**z_quad for " + str(len(unknown_ids)) + " unknown-z targets:** "
            + ", ".join(str(m) for m in unknown_ids)
        ))
    for msa_id in unknown_ids:
        grating_files = target_groups[msa_id]
        fpath         = grating_files[0]
        global_idx    = files.index(fpath)
        combined, result_1d = _load_spectrum(fpath)
        if result_1d is None:
            display(Markdown("⚠️ MSA " + str(msa_id) + ": no spectrum \u2014 skipped."))
            continue
        wave, flux, err = result_1d
        good = np.isfinite(wave) & np.isfinite(flux)
        if err is not None:
            good &= np.isfinite(err) & (err > 0)
        if good.sum() < 3:
            continue
        w, f_arr  = wave[good], flux[good]
        e         = err[good] if err is not None else None
        suptitle, subtitle = build_title(combined, fpath)
        hdr_z = extract_redshift_from_header(combined)

        fig, axes_2x2 = plt.subplots(
            2, 2, figsize=(FIG_WIDTH, PANEL_HEIGHT * 2.1), facecolor="#0d0d1a"
        )
        fig.patch.set_facecolor("#0d0d1a")
        z_hdr_str = ("  |  header z = " + f"{hdr_z:.4f}") if hdr_z is not None else ""
        fig.suptitle(
            "MSA " + str(msa_id) + "  |  " + suptitle + z_hdr_str + "\n"
            "z unknown \u2014 four redshift windows for identification",
            color="white", fontsize=9, fontweight="bold", y=0.995, va="top",
        )
        for quad_idx, (z_label, z_val) in enumerate(Z_WINDOWS):
            ax = axes_2x2[quad_idx // 2][quad_idx % 2]
            _draw_1d_panel(
                ax=ax, w=w, f_arr=f_arr, e=e,
                redshift=z_val, z_label=z_label,
                global_idx=global_idx, fpath=fpath,
                suptitle=suptitle, subtitle=subtitle,
                col=quad_idx % 2, active_groups=active_groups,
            )
        if active_groups:
            _make_line_legend(fig, active_groups)
        plt.tight_layout(rect=[0, 0.06 if active_groups else 0.02, 1, 0.93])
        if save:
            _OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
            out = _OUTPUT_DIR / ("glass_jwst_zquad_" + str(msa_id) + ".png")
            fig.savefig(out, dpi=150, bbox_inches="tight",
                        facecolor="#0d0d1a", pad_inches=0.08)
            display(Markdown("💾 MSA " + str(msa_id) + " z_quad saved → `" + str(out.resolve()) + "`"))
        plt.show()


# ---------------------------------------------------------------------------
#  Dispatcher
# ---------------------------------------------------------------------------

if spec_files:
    if IDX_START > IDX_END:
        IDX_START, IDX_END = IDX_END, IDX_START

    _end_clamped = min(IDX_END, len(spec_files) - 1)
    if _end_clamped < IDX_END:
        display(Markdown(
            "ℹ️ `IDX_END` clamped " + str(IDX_END) + " \u2192 " + str(_end_clamped) +
            " (only " + str(len(spec_files)) + " files available)."
        ))

    if PLOT_MODE == "auto":
        _run_auto(
            files       = spec_files,
            idx_start   = IDX_START,
            idx_end     = _end_clamped,
            show_lines  = SHOW_LINES,
            line_groups = ANNOTATE_GROUPS,
            save        = SAVE_PNG,
        )
    elif PLOT_MODE == "z_quad":
        plot_z_quad(
            files       = spec_files,
            idx_start   = IDX_START,
            idx_end     = _end_clamped,
            z_windows   = Z_WINDOWS,
            show_lines  = SHOW_LINES,
            line_groups = ANNOTATE_GROUPS,
            save        = SAVE_PNG,
        )
    else:
        plot_range(
            files       = spec_files,
            idx_start   = IDX_START,
            idx_end     = _end_clamped,
            show_2d     = SHOW_2D,
            show_lines  = SHOW_LINES,
            line_groups = ANNOTATE_GROUPS,
            save        = SAVE_PNG,
        )
else:
    display(Markdown("⚠️ No spectra loaded \u2014 check `MAST_ROOT` and re-run Cell 3."))
